# 🎙️ VoiceCloner — Viterbox TTS trên Colab

Repo: https://github.com/kanazawahere/VoiceCloner

**Tính năng:**
- ✅ Python 3.12 compatible (numpy>=1.26)
- ✅ HuggingFace cache lưu vào Google Drive
- ✅ Zero-shot voice cloning
- ✅ Xử lý văn bản dài

In [ ]:
# ✅ BƯỚC 1: Trỏ HuggingFace cache về Drive
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)

os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

print(f"✅ HF cache: {DRIVE_HF_CACHE}")

In [ ]:
# ✅ BƯỚC 2: Cài uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
current_path = os.environ.get('PATH', '')
new_path = f"/root/.local/bin:{current_path}"
os.environ['PATH'] = new_path
%env PATH={new_path}
!uv --version

In [ ]:
# ✅ BƯỚC 3: Clone repo
import shutil
if os.path.exists('/content/VoiceCloner'):
    shutil.rmtree('/content/VoiceCloner')

!git clone https://github.com/kanazawahere/VoiceCloner.git \
    /content/VoiceCloner --depth=1 --quiet
%cd /content/VoiceCloner
print('✅ Clone xong')

In [ ]:
# ✅ BƯỚC 4: Cài packages
import subprocess
cuda_ver = subprocess.getoutput(
    r"nvcc --version | grep 'release' | sed 's/.*release //' | sed 's/,.*//' | sed 's/\.//'"
).strip()
if not cuda_ver.isdigit():
    cuda_ver = "121"
torch_index = f"https://download.pytorch.org/whl/cu{cuda_ver}"
print(f"CUDA: cu{cuda_ver}")

# Cài torch GPU TRƯỚC
!uv pip install --system torch torchvision torchaudio \
    --index-url {torch_index} --quiet

# Cài requirements (bỏ qua torch/torchaudio)
!uv pip install --system -r requirements.txt --quiet
!uv pip install --system -e . --quiet

print("✅ Cài xong")
print("⚠️ QUAN TRỌNG: Chạy cell tiếp theo để restart runtime!")

In [ ]:
# ✅ BƯỚC 4.5: Restart runtime để load lại numpy
import os
os.kill(os.getpid(), 9)

In [ ]:
# ✅ BƯỚC 5: Kiểm tra môi trường sau restart
import os
print(f"HF_HOME: {os.environ.get('HF_HOME', 'NOT SET')}")
print(f"Working dir: {os.getcwd()}")

# Set lại env vars nếu cần
if 'HF_HOME' not in os.environ:
    DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
    os.environ['HF_HOME'] = DRIVE_HF_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE
    print(f"✅ Set HF_HOME: {DRIVE_HF_CACHE}")

# Chuyển về thư mục VoiceCloner
if not os.path.exists('/content/VoiceCloner'):
    print("⚠️ Thư mục VoiceCloner không tồn tại. Chạy lại từ BƯỚC 3!")
else:
    %cd /content/VoiceCloner
    print("✅ Sẵn sàng")

In [ ]:
# ✅ BƯỚC 6: Load model
import torch
from viterbox import Viterbox

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tts = Viterbox.from_pretrained(device)
print("✅ Model load xong!")

In [ ]:
# ✅ BƯỚC 7: Tổng hợp giọng nói
from IPython.display import Audio, display

text = "Đây là bài kiểm tra hệ thống chuyển văn bản thành giọng nói tiếng Việt."

audio = tts.generate(text=text, language="vi")
tts.save_audio(audio, "/content/output.wav")
display(Audio("/content/output.wav"))
print("✅ Done!")